In [2]:
# carga la extensión autoreload
%load_ext autoreload

# establece el nivel de autorecarga en 2
%autoreload 2

import sys
sys.path.append('../')

from db.client import MongoDBConnection

In [3]:
import fitz
from google import genai
from PIL import Image
import io
import pandas as pd

from pydantic import BaseModel
from typing import List, Dict, Optional
from pydantic import Field
import traceback
import os

### CONEXIÓN A LA BASE DE DATOS

In [98]:
mongo_connection = MongoDBConnection()
db = mongo_connection.get_database()

In [99]:
collection = db["operaciones"]

In [4]:
archivo_pdf = "datos/Siplan- normativa.pdf"
client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [101]:
prompt = """
Extrae todas las tablas visibles en la imagen.

Cada tabla corresponde a una operación.
Cada fila corresponde a una acción.

No cambies el nombre de ninguna clave.
No omitas ninguna de las 8 claves.
Si un valor no existe usa null.
No agregues campos adicionales.
No incluyas texto fuera del JSON.
"""

In [102]:
class Trimestres(BaseModel):
    T1_porcentaje: Optional[str]
    T1_cantidad: Optional[str]
    T2_porcentaje: Optional[str]
    T2_cantidad: Optional[str]
    T3_porcentaje: Optional[str]
    T3_cantidad: Optional[str]
    T4_porcentaje: Optional[str]
    T4_cantidad: Optional[str]

class Accion(BaseModel):
    accion: str = Field(description="Texto completo de la acción")
    producto: str = Field(description="Producto o entregable asociado a la acción")
    meta: str = Field(description="Meta establecida para la acción")
    avance_periodo_actual: str = Field(description="Porcentaje de avance en el periodo actual")
    trimestres: Trimestres = Field(description="Avance distribuido por trimestres")
    responsable: str = Field(description="Nombre completo del responsable")
    total_avance_por_accion: str = Field(description="Porcentaje total acumulado de avance")
    descripcion: Optional[str] = Field(
        description="Observaciones o descripción del avance por trimestre"
    )

class TablaOperacion(BaseModel):
    politica: str = Field(description="Número o código de política")
    proceso: str = Field(description="Código y nombre del proceso")
    operacion: str = Field(description="Código y nombre de la operación")
    avance_operacion: str = Field(description="Porcentaje total de avance de la operación")
    acciones: List[Accion] = Field(description="Lista de acciones asociadas a la operación")

class OperacionesResponse(BaseModel):
    operaciones: List[TablaOperacion]

In [103]:
doc = fitz.open(archivo_pdf)

In [104]:
for page_index in range(len(doc)):

    print(f"\n📄 Procesando página {page_index + 1}...")

    try:
        page = doc.load_page(page_index)

        # 🔹 Renderizar rotado 90° derecha
        mat = fitz.Matrix(2, 2).prerotate(90)
        pix = page.get_pixmap(matrix=mat)

        img_bytes = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_bytes))

        # 🔹 Llamada a Gemini
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[prompt, image],
            config={
                "response_mime_type": "application/json",
                "response_schema": OperacionesResponse
            }
        )
        
        print("Tokens usados:")
        print(response.usage_metadata)

        data: OperacionesResponse = response.parsed

        # 🔹 Insertar en MongoDB
        for operacion in data.operaciones:
            documento = operacion.model_dump()
            documento["archivo_origen"] = archivo_pdf
            documento["pagina_origen"] = page_index + 1

            collection.insert_one(documento)

        print(f"✅ Página {page_index + 1} insertada correctamente.")

    except Exception as e:
        print(f"❌ Error en página {page_index + 1}")
        print(str(e))
        traceback.print_exc()
        continue

print("\n🚀 Proceso finalizado.")


📄 Procesando página 1...
Tokens usados:
cache_tokens_details=None cached_content_token_count=None candidates_token_count=2803 candidates_tokens_details=None prompt_token_count=334 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=76
), ModalityTokenCount(
  modality=<MediaModality.IMAGE: 'IMAGE'>,
  token_count=258
)] thoughts_token_count=3003 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=6140 traffic_type=None
✅ Página 1 insertada correctamente.

📄 Procesando página 2...
Tokens usados:
cache_tokens_details=None cached_content_token_count=None candidates_token_count=1395 candidates_tokens_details=None prompt_token_count=334 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=76
), ModalityTokenCount(
  modality=<MediaModality.IMAGE: 'IMAGE'>,
  token_count=258
)] thoughts_token_count=2800 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=

In [105]:
p

NameError: name 'p' is not defined

In [ ]:
for i, page in enumerate(doc):

In [ ]:
# 👉 SOLO PRIMERA PÁGINA
page = doc.load_page(0)

# 🔹 Renderizar rotado 90° derecha
mat = fitz.Matrix(2, 2).prerotate(90)
pix = page.get_pixmap(matrix=mat)

# Guardar imagen para validar visualmente
pix.save("pagina_1_rotada.png")

img_bytes = pix.tobytes("png")
image = Image.open(io.BytesIO(img_bytes))

prompt = """
Extrae TODAS las tablas visibles en esta imagen.

Devuelve únicamente JSON válido.
Si hay múltiples tablas, devuélvelas como un array.
Mantén todos los encabezados completos.
No inventes datos.
Si una celda está vacía usa null.

Estructura esperada:
{
  "politica": "",
  "proceso": "",
  "operacion": "",
  "avance_operacion": "",
  "acciones": [
    {
      "accion": "",
      "producto": "",
      "meta": "",
      "trimestres": {},
      "responsable": "",
      "total_avance_por_accion": "",
      "descripcion": ""
    }
  ]
}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",  # nombre en minúscula
    contents=[prompt, image],
    config={
        "response_mime_type": "application/json"
    }
)

# 🔹 Guardar respuesta
try:
    data = json.loads(response.text)

    with open("resultado_pagina_1.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print("✅ JSON guardado en resultado_pagina_1.json")

except Exception as e:
    print("⚠️ No fue JSON válido. Guardando texto crudo.")
    
    with open("resultado_pagina_1_raw.txt", "w", encoding="utf-8") as f:
        f.write(response.text)

    print("⚠️ Se guardó respuesta cruda en resultado_pagina_1_raw.txt")

⚠️ No fue JSON válido. Guardando texto crudo.
⚠️ Se guardó respuesta cruda en resultado_pagina_1_raw.txt


In [ ]:
data

NameError: name 'data' is not defined